# Fine-tune a Family Request Router (Qwen3)

**Week 5 project — custom-dataset variant.** Run this on a **free Tesla T4 Colab GPU**.

This follows the same recipe as the reference project (fine-tune `Qwen/Qwen3-1.7B-Base`
with a LoRA adapter through the [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory)
visual UI), applied to a domain I actually own: routing requests inside my
[family-calendar](https://github.com/anushaakkiraju26/family-calendar) Deep Agent project
instead of IT support tickets.

Dataset: `data/family_request_routing.csv` in this repo — 30 real, eval-labelled parent
requests from family-calendar's evaluation suite, expanded with synthetic examples via
`tools/generate_dataset.py` to ~50 balanced examples per label. See that script and
`data/family_request_routing_manifest.json` for exact provenance.

---
## The scenario

The Family Coordinator in my project is a Deep Agent: every incoming parent request first
gets reasoned about by a large hosted model to decide *how* to handle it — a single direct
tool call, an outright rejection, a full multi-agent weekly-planning workflow, a clarifying
question, or an outing-research call. That routing decision happens on **every** request,
before any real work starts, and today it costs a full frontier-model call each time.

```
fast_path_mutate      -> direct create/update/move/delete/restore, pending approval
fast_path_read        -> direct list/show, no mutation
fast_path_reject      -> deterministically declined (past event, conflict, cross-family, ...)
deep_weekly_workflow  -> full specialist pipeline: intake, planner, transportation, review, reminders
outing_workflow       -> Family Outing Agent's constrained search wrapper
ambiguous_clarify     -> missing info, must ask before acting
```

The model never plans the week or drafts a reminder itself — it just predicts which of these
six paths a request belongs to, the same way the reference project's router predicts a
support-ticket queue. The rest of the coordinator's existing tools and specialists take over
from there.

**Why not just call the frontier model for this every time?** That's the actual production
setup right now, and it works — but a 7-way (well, 6-way) routing decision on a short message
does not need a frontier model's full reasoning. A small fine-tuned classifier can make the
same call in milliseconds on modest hardware, and only escalate to the full agent once routing
is already decided.

## 1. Install dependencies

In [ ]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]

### Check GPU environment

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: Runtime > Change runtime type > T4 GPU")

## 2. Prepare the family-request routing dataset (INPUT REQUIRED)

The labelled CSV already lives in this repo, so the simplest path is to clone it
straight into Colab. If the repo isn't reachable (private, not pushed yet, etc.), this
falls back to a manual upload widget for `family_request_routing.csv`.

This cell filters to the six known labels, creates a **stratified 80/20 train/val split**,
converts the training split to **ShareGPT JSON**, writes it to
`/content/LLaMA-Factory/data/TRAIN.json`, and registers it in `dataset_info.json` under the
name `family_request_routing` so it shows up in the LLaMA Board UI.

In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# ── constants ────────────────────────────────────────────────────────────────
LLAMA_DATA_DIR  = "/content/LLaMA-Factory/data"
TRAIN_JSON_PATH = f"{LLAMA_DATA_DIR}/TRAIN.json"
DATASET_INFO    = f"{LLAMA_DATA_DIR}/dataset_info.json"
REPO_URL        = "https://github.com/anushaakkiraju26/family-request-router.git"
REPO_DIR        = "/content/family-request-router"

LABEL2ID = {
    "fast_path_mutate":     0,
    "fast_path_read":       1,
    "fast_path_reject":     2,
    "deep_weekly_workflow": 3,
    "outing_workflow":      4,
    "ambiguous_clarify":    5,
}

# Single source of truth for label strings + their human-readable names —
# every later cell (baseline eval, classify(), confusion matrix, charts)
# reads LABEL_TOKENS / LABEL_DISPLAY / display_labels from here rather than
# redeclaring them, so there's one place to edit if labels ever change.
LABEL_TOKENS = list(LABEL2ID)
LABEL_DISPLAY = {
    "fast_path_mutate":     "Schedule change",
    "fast_path_read":       "List / show",
    "fast_path_reject":     "Declined",
    "deep_weekly_workflow": "Weekly plan",
    "outing_workflow":      "Outing search",
    "ambiguous_clarify":    "Needs clarification",
}
display_labels = [LABEL_DISPLAY[l] for l in LABEL_TOKENS]

# v2 system prompt — same 321-row dataset as the first pass, but with explicit
# disambiguating detail for the label pairs that collapsed in that run's
# confusion matrix (fast_path_reject swallowing ambiguous_clarify and
# deep_weekly_workflow; outing_workflow being misread as fast_path_read).
# This exact string must also be used at inference time in cell 16's
# classify() — a mismatch there would confound the experiment.
SYSTEM_PROMPT = (
    "You are a family-calendar coordinator's routing assistant. Given a parent's "
    "request, respond with exactly one of the following six categories:\n"
    "- fast_path_mutate: a single, specific create/update/move/delete/restore of "
    "one event or reminder, pending approval. Use this for ordinary cancel/change/"
    "add requests with no past-dated event, no stated conflict, and no "
    "cross-family issue.\n"
    "- fast_path_read: a single, specific read-only list/show request for events "
    "already on the calendar. No mutation.\n"
    "- fast_path_reject: use ONLY when the request itself states an explicit "
    "disqualifying fact — a clearly past-dated event, a stated conflict, "
    "cross-family access, or a repeat of something already rejected. Do not use "
    "this as a default when merely uncertain.\n"
    "- deep_weekly_workflow: a broad request spanning multiple days or a whole "
    "week, not one event.\n"
    "- outing_workflow: a request to search for or suggest NEW outing/activity "
    "ideas — not to list or show events already on the calendar.\n"
    "- ambiguous_clarify: use ONLY when a specific required detail is missing "
    "(which event, which family, which time) and no other label clearly fits. Do "
    "not use this as a default when merely uncertain either.\n"
    "Respond with exactly one label and nothing else."
)

# ── 1. Get the CSV: clone this repo, or fall back to manual upload ────────────
csv_path = Path(REPO_DIR) / "data" / "family_request_routing.csv"
if not csv_path.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=False)

if csv_path.exists():
    CSV_PATH = str(csv_path)
    print(f"Using cloned dataset: {CSV_PATH}")
else:
    from google.colab import files
    print("Repo clone unavailable — upload family_request_routing.csv manually:")
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]

# ── 2. Load + filter ────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH).rename(columns={"category_truth": "label"})
df = df[df["label"].isin(LABEL2ID)].sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Loaded {len(df):,} rows")
print(df["label"].value_counts())

# ── 3. Stratified train/val split ───────────────────────────────────────────
df_train, df_val = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42,
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
print(f"\nTrain: {len(df_train):,} rows | Val (held-out): {len(df_val):,} rows")
print("\nTrain label distribution:")
print(df_train["label"].value_counts())

# ── 4. Convert train split -> ShareGPT JSON ─────────────────────────────────
sharegpt_records = [
    {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"Parent request: {row['text']}"},
            {"role": "assistant", "content": row["label"]},
        ]
    }
    for _, row in df_train.iterrows()
]

with open(TRAIN_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(sharegpt_records, f, indent=2, ensure_ascii=False)
print(f"\nWritten {len(sharegpt_records):,} records -> {TRAIN_JSON_PATH}")

# ── 5. Register dataset in dataset_info.json ────────────────────────────────
with open(DATASET_INFO, "r", encoding="utf-8") as f:
    info = json.load(f)

info["family_request_routing"] = {
    "file_name": "TRAIN.json",
    "formatting": "sharegpt",
    "columns":   {"messages": "messages"},
    "tags": {
        "role_tag":      "role",
        "content_tag":   "content",
        "user_tag":      "user",
        "assistant_tag": "assistant",
        "system_tag":    "system",
    },
}

with open(DATASET_INFO, "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)
print(f"Registered 'family_request_routing' in {DATASET_INFO}")

# ── 6. Save val split for evaluation ────────────────────────────────────────
VAL_CSV = "/content/val_split.csv"
df_val.to_csv(VAL_CSV, index=False)
print(f"Val split saved -> {VAL_CSV}  ({len(df_val):,} rows)")

## 3. Fine-tune via LLaMA Board (INPUT REQUIRED)

1. Run the next cell to start the LLaMA Board server, then open the **public** URL it prints.
2. Base model: `Qwen/Qwen3-1.7B-Base`. Dataset: `family_request_routing`. Finetuning: **LoRA**.
   Defaults are fine for a first run.
3. When training finishes, note the **Output Dir** path (`train_2026-...`) — you'll need it below.
4. **Manually stop this cell** once training completes — Colab won't stop the server for you.

Watch for the "training completed" message in the UI or the logs. Occasional "syntax error"
toasts from LLaMA Board are a known cosmetic issue and don't affect training. A short LoRA run
on ~250 training rows usually takes well under the reference project's 30–60 min estimate.

### Hyperparameters, briefly

Defaults are fine for a first pass. If you need to adjust:

| Knob | Effect | If your run is off |
|---|---|---|
| Learning rate | step size per update | loss oscillating -> lower it; loss barely moving -> raise it carefully |
| Epochs | passes over training data | loss still falling at the end -> add an epoch; val score drops late -> stop earlier |
| Batch size | examples per gradient step | tune for GPU memory / gradient noise |
| LoRA rank | adapter capacity | raise only if the task clearly needs more expressiveness — six labels rarely do |

In [ ]:
%cd /content/LLaMA-Factory/
!GRADIO_SHARE=1 llamafactory-cli webui

---
## 4. Review training — loss curve (INPUT REQUIRED)

Set `ADAPTER_DIR` to the Output Dir path from LLaMA Board. The loss should drop and level
off; flat or chaotic usually means the learning rate, dataset size, or chat template is off.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

ADAPTER_DIR = "/content/LLaMA-Factory/saves/Qwen3-1.7B-Base/lora/train_XXXX-XX-XX-XX-XX-XX"  # <-- CHANGE THIS
MERGED_DIR      = "/content/family_router_merged"
BASE_MODEL_NAME = "Qwen/Qwen3-1.7B-Base"

log_file = Path(ADAPTER_DIR) / "trainer_log.jsonl"
if not log_file.exists():
    raise FileNotFoundError(f"trainer_log.jsonl not found in {ADAPTER_DIR!r}")

records = [json.loads(l) for l in log_file.read_text().splitlines() if l.strip()]
steps   = [r["current_steps"] for r in records if r.get("loss") is not None]
losses  = [r["loss"]          for r in records if r.get("loss") is not None]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, losses, linewidth=1.5, color="#1565C0", alpha=0.85)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training loss curve")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/content/training_curve.png", dpi=120)
plt.show()

total_drop = losses[0] - losses[-1]
print(f"Starting loss : {losses[0]:.4f}")
print(f"Final loss    : {losses[-1]:.4f}")
print(f"Total drop    : {total_drop:.4f}")
print()
if losses[-1] < 0.5:
    print("Loss is low - model has likely converged well.")
elif losses[-1] < 1.2:
    print("Loss is moderate - model has learned but may benefit from more epochs.")
else:
    print("Loss is still high - consider more epochs, a lower learning rate, or checking the data format.")

---
## 5. Merge the adapter and measure a baseline

Training kept the base weights frozen and only trained a small LoRA adapter. Merging folds
those deltas into the base weights, giving one standalone model with no adapter overhead.

The baseline below is the *same* base model, on the *same* validation set, with **no**
fine-tuning — a constrained six-letter multiple-choice prompt so it can't fail just because it
phrases a label slightly differently. We measure it now, before attaching the adapter, so the
base model doesn't need to be loaded twice.

In [ ]:
import gc
import os
from pathlib import Path

import pandas as pd
import safetensors.torch as st
import torch
from peft import PeftConfig, get_peft_model
from peft.utils import set_peft_model_state_dict
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── USER INPUT ───────────────────────────────────────────────────────────────
# LABEL_TOKENS / LABEL_DISPLAY / display_labels come from cell 7.
CHOICES      = "ABCDEF"
CHOICE2LABEL = {ch: lbl for ch, lbl in zip(CHOICES, LABEL_TOKENS)}

BASE_SYSTEM_PROMPT = (
    "You are a family-calendar coordinator's routing assistant. "
    "Classify the parent request by responding with ONLY a single letter - nothing else:\n"
    + "\n".join(f"{ch}) {lbl}" for ch, lbl in CHOICE2LABEL.items())
)

# ── Validate adapter ─────────────────────────────────────────────────────────
adapter_path = Path(ADAPTER_DIR)
if not adapter_path.is_dir():
    raise FileNotFoundError(f"Adapter folder not found: {ADAPTER_DIR!r}")
if not (adapter_path / "adapter_config.json").exists():
    raise FileNotFoundError(f"No adapter_config.json in {ADAPTER_DIR!r}")
print(f"Adapter found: {ADAPTER_DIR}")

# ── Device ───────────────────────────────────────────────────────────────────
HAS_CUDA = torch.cuda.is_available()
HAS_MPS  = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
DEVICE   = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")
dtype    = torch.float16 if DEVICE in ("cuda", "mps") else torch.float32
print(f"Device: {DEVICE}")

# ── Load base model + tokenizer ─────────────────────────────────────────────
print("\nLoading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=dtype, device_map=DEVICE,
)
tok_src    = str(adapter_path) if (adapter_path / "tokenizer.json").exists() else BASE_MODEL_NAME
_tokenizer = AutoTokenizer.from_pretrained(tok_src)
base_model.eval()

# ── Baseline inference on val split ─────────────────────────────────────────
_choice_ids = []
for ch in CHOICES:
    ids_plain  = _tokenizer.encode(ch,       add_special_tokens=False)
    ids_spaced = _tokenizer.encode(f" {ch}", add_special_tokens=False)
    _choice_ids.append(ids_spaced[0] if len(ids_spaced) == 1 else ids_plain[0])


def classify_base(request_text: str) -> str:
    messages = [
        {"role": "system", "content": BASE_SYSTEM_PROMPT},
        {"role": "user",   "content": f"Parent request: {request_text}"},
    ]
    prompt = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _tokenizer(prompt, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model(input_ids=inputs["input_ids"])
    first_logits = out.logits[0, -1, :]
    choice_probs = torch.softmax(first_logits[_choice_ids], dim=-1).cpu().tolist()
    return CHOICE2LABEL[CHOICES[choice_probs.index(max(choice_probs))]]


df_val      = pd.read_csv("/content/val_split.csv")
y_true      = df_val["label"].tolist()
y_pred_base = []

for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Baseline inference"):
    y_pred_base.append(classify_base(row["text"]))

from sklearn.metrics import classification_report
print("\n=== Baseline (no fine-tuning) ===")
print(classification_report(y_true, y_pred_base, target_names=display_labels, digits=3, zero_division=0))

# ── Attach LoRA + load weights ───────────────────────────────────────────────
print("Attaching LoRA adapter...")
peft_cfg   = PeftConfig.from_pretrained(str(adapter_path))
peft_model = get_peft_model(base_model, peft_cfg)

candidates = [
    adapter_path / "adapter_model.safetensors",
    adapter_path / "adapters.safetensors",
    adapter_path / "adapter_model.bin",
]
weights_file = next((p for p in candidates if p.exists()), None)
if weights_file is None:
    raise FileNotFoundError(f"No adapter weights in {ADAPTER_DIR}")

if weights_file.suffix == ".safetensors":
    adapter_weights = st.load_file(str(weights_file), device=DEVICE)
else:
    adapter_weights = torch.load(str(weights_file), map_location=DEVICE)

set_peft_model_state_dict(peft_model, adapter_weights)

# ── Merge + save to disk ─────────────────────────────────────────────────────
print("Merging (this may take a while) ...")
_model = peft_model.merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
_model.save_pretrained(MERGED_DIR)
_tokenizer.save_pretrained(MERGED_DIR)
stale = Path(MERGED_DIR) / "adapter_config.json"
if stale.exists():
    stale.unlink()
print(f"Saved -> {MERGED_DIR}")

del peft_model, base_model, adapter_weights
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

_model.eval()
print("Model ready for inference.")

---
## 6. `classify()` and a smoke test

Chat-format the request with the same system prompt used in training, generate a few tokens,
and match the start of the output to one of the six labels. `classify()` also reports a
confidence score — the softmax probability of the winning label's first token versus the
other five. Five obvious requests, one per class, should all route correctly before trusting
the full validation run.

In [ ]:
import torch

# Must exactly match the SYSTEM_PROMPT used to build the training data in
# cell 7 — this is the v2, more-detailed prompt (same 321-row dataset,
# richer per-label disambiguation). A mismatch here would train on one
# framing and query with another, invalidating the comparison to pass 1.
SYSTEM_PROMPT = (
    "You are a family-calendar coordinator's routing assistant. Given a parent's "
    "request, respond with exactly one of the following six categories:\n"
    "- fast_path_mutate: a single, specific create/update/move/delete/restore of "
    "one event or reminder, pending approval. Use this for ordinary cancel/change/"
    "add requests with no past-dated event, no stated conflict, and no "
    "cross-family issue.\n"
    "- fast_path_read: a single, specific read-only list/show request for events "
    "already on the calendar. No mutation.\n"
    "- fast_path_reject: use ONLY when the request itself states an explicit "
    "disqualifying fact — a clearly past-dated event, a stated conflict, "
    "cross-family access, or a repeat of something already rejected. Do not use "
    "this as a default when merely uncertain.\n"
    "- deep_weekly_workflow: a broad request spanning multiple days or a whole "
    "week, not one event.\n"
    "- outing_workflow: a request to search for or suggest NEW outing/activity "
    "ideas — not to list or show events already on the calendar.\n"
    "- ambiguous_clarify: use ONLY when a specific required detail is missing "
    "(which event, which family, which time) and no other label clearly fits. Do "
    "not use this as a default when merely uncertain either.\n"
    "Respond with exactly one label and nothing else."
)
# LABEL_TOKENS / LABEL_DISPLAY / display_labels come from cell 7.


def classify(request_text: str, compute_confidence: bool = True) -> tuple:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Parent request: {request_text}"},
    ]
    prompt = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _tokenizer(prompt, return_tensors="pt").to(_model.device)

    with torch.no_grad():
        out = _model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=_tokenizer.eos_token_id,
        )

    generated = _tokenizer.decode(
        out.sequences[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()

    matched = next((l for l in LABEL_TOKENS if generated.lower().startswith(l.lower())), None)
    if matched is None:
        matched = "ambiguous_clarify"

    if not compute_confidence or not out.scores:
        return matched, 1.0

    first_scores = out.scores[0][0]
    probs = torch.softmax(first_scores, dim=-1)
    confidence = probs.max().item()
    return matched, confidence


SMOKE_TESTS = [
    ("Add Maya's swim class tomorrow from 4 to 5 PM for family-1", "fast_path_mutate"),
    ("Show today's activities for family-2", "fast_path_read"),
    ("Add Leo's soccer practice yesterday from 4 to 5 PM for family-1", "fast_path_reject"),
    ("Coordinate next week for family-1, resolve conflicts, and propose parent assignments", "deep_weekly_workflow"),
    ("Find three outdoor activities near San Jose for family-1 this weekend", "outing_workflow"),
    ("Delete soccer practice for family-1", "ambiguous_clarify"),
]

correct = 0
for text, expected in SMOKE_TESTS:
    pred, conf = classify(text)
    ok = "OK" if pred == expected else "MISMATCH"
    correct += pred == expected
    print(f"[{ok:8s}] expected={expected:22s} predicted={pred:22s} conf={conf:.1%}  | {text}")

print(f"\n{correct}/{len(SMOKE_TESTS)} smoke tests passed.")

---
## 7. Evaluate on the held-out validation split

These rows were never seen during training, so this is the honest read on routing accuracy.

In [ ]:
from tqdm.auto import tqdm

y_pred = []
for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Fine-tuned inference"):
    pred, _ = classify(row["text"], compute_confidence=False)
    y_pred.append(pred)

print(f"Evaluated {len(y_pred):,} samples.")

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=display_labels, digits=3))

print()
for i in [0, 5, 10, 15]:
    if i >= len(df_val):
        continue
    row  = df_val.iloc[i]
    pred, conf = classify(row["text"])
    print(f"=== Request {i}")
    print(f"TRUE label: {row['label']}")
    print(f"PRED label: {pred}  (conf {conf:.1%})")
    print(f"Text: {row['text']}")
    print()

### Reading the report

`fast_path_reject` and `fast_path_mutate` recall matter most here: a missed rejection means an
invalid mutation (a past event, a real conflict) slips further into the pipeline than it
should before the deterministic safeguards in `repository.py` catch it. A missed
`deep_weekly_workflow` just costs a slower, single-turn answer to what should have been a
weekly plan — annoying, not unsafe.

### Confusion matrix

Rows are true labels, columns are predictions; the diagonal is correct. Colour is
row-normalised so it doubles as a recall heatmap. Watch for `fast_path_reject` rows leaking
into `fast_path_mutate` columns — that's the confusion pair worth the most attention.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm      = confusion_matrix(y_true, y_pred, labels=LABEL_TOKENS)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_norm, annot=cm, fmt="d", cmap="Blues",
    xticklabels=display_labels, yticklabels=display_labels,
    linewidths=0.5, ax=ax,
)
ax.set_title("Confusion matrix (counts shown, colour = row-normalised recall)")
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("/content/confusion_matrix.png", dpi=120)
plt.show()

reject_recall = cm_norm[LABEL_TOKENS.index("fast_path_reject"), LABEL_TOKENS.index("fast_path_reject")]
print(f"\nfast_path_reject recall: {reject_recall:.1%}")
if reject_recall < 0.85:
    print("Warning: reject recall below 0.85 - consider more training data for this class.")

---
## 8. Baseline vs fine-tuned

The baseline is the identical base model, same validation requests, zero task-specific
training — its best shot via the constrained six-letter prompt. The gap between it and the
fine-tuned model is the measurable value of this labelled dataset and this training run.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report


def per_class_f1(y_t, y_p, labels):
    report = classification_report(y_t, y_p, target_names=labels, output_dict=True, zero_division=0)
    return {lbl: report[lbl]["f1-score"] for lbl in labels}


ft_f1    = per_class_f1(y_true, y_pred,      LABEL_TOKENS)
base_f1  = per_class_f1(y_true, y_pred_base, LABEL_TOKENS)
ft_acc   = accuracy_score(y_true, y_pred)
base_acc = accuracy_score(y_true, y_pred_base)

labels_plot = display_labels + ["overall accuracy"]
ft_vals     = [ft_f1[l]   for l in LABEL_TOKENS] + [ft_acc]
base_vals   = [base_f1[l] for l in LABEL_TOKENS] + [base_acc]

x     = np.arange(len(labels_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars_base = ax.bar(x - width/2, base_vals, width, label="Base model (no fine-tuning)", color="#90CAF9", edgecolor="white")
bars_ft   = ax.bar(x + width/2, ft_vals,   width, label="Fine-tuned (LLaMA Board LoRA)", color="#1565C0", edgecolor="white")

ax.bar_label(bars_base, fmt="{:.2f}", padding=3, fontsize=8)
ax.bar_label(bars_ft,   fmt="{:.2f}", padding=3, fontsize=8)
bars_base[-1].set_color("#FFCC80")
bars_ft[-1].set_color("#E65100")

ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels([l.replace(" ", "\n") for l in labels_plot], fontsize=9)
ax.set_ylabel("F1 score / Accuracy")
ax.set_title("Family request router - baseline vs fine-tuned")
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.15), ncol=2)
plt.tight_layout()
plt.savefig("/content/baseline_vs_finetuned.png", dpi=120)
plt.show()

print(f"Baseline accuracy    : {base_acc:.1%}")
print(f"Fine-tuned accuracy  : {ft_acc:.1%}")
print(f"Delta                : {(ft_acc - base_acc) * 100:+.1f} pts")

## Recap

- Fine-tuned `Qwen/Qwen3-1.7B-Base` with a LoRA adapter (rank 8, 3 epochs) through LLaMA Board
  on `data/family_request_routing.csv` — 30 real, eval-labelled parent requests (sourced from
  [family-calendar](https://github.com/anushaakkiraju26/family-calendar)'s evaluation suite)
  expanded to 321 balanced examples across 6 routing labels via `tools/generate_dataset.py`.
- Compared the merged fine-tuned model against a constrained-choice baseline on a 65-row
  held-out validation split, across two training passes that isolate one variable each.

### Results — two passes, same data, same split, same LoRA config

| Metric | Baseline | Pass 1 (minimal prompt) | Pass 2 (richer per-label prompt) |
|---|---|---|---|
| Overall accuracy | 24.6% | 36.9% (+12.3 pts) | **46.2%** (+21.5 pts) |

Per-class recall:

| Label | Pass 1 | Pass 2 | Change |
|---|---|---|---|
| `fast_path_read` | 100% | 100% | — |
| `outing_workflow` | 18.2% | **72.7%** | **+54.5 pts** |
| `fast_path_mutate` | 63.6% | 54.5% | −9.1 pts |
| `fast_path_reject` | 36.4% | 36.4% | 0 |
| `deep_weekly_workflow` | 0% | 10% | +10 pts |
| `ambiguous_clarify` | 0% | **0%** | 0 |

Pass 1 used a minimal one-line system prompt (just the six label names). Pass 2 used the exact
same 321-row dataset and the exact same LoRA config, changing only the `SYSTEM_PROMPT` used to
build the training data (and, matched at inference time) to include per-label definitions and
explicit "do not use this as a default when merely uncertain" guidance for the two classes
that were behaving like a catch-all.

### Finding: prompt instructions fix *lexical* confusion, not *reasoning* confusion

The `outing_workflow` ↔ `fast_path_read` confusion **resolved dramatically** with prompt
changes alone (18.2% → 72.7% recall) — that pair is essentially lexical: "search for new
activities" vs. "list existing events" is a distinction the model could apply correctly once
told explicitly which cue to attend to.

The `fast_path_reject` cluster **did not move at all**. `ambiguous_clarify` recall stayed at
0% — all 11 held-out examples still route to `fast_path_reject` in both passes — despite the
pass 2 prompt explicitly instructing *both* labels not to be used as an uncertainty default.
`fast_path_reject`'s own recall was identically 36.4% in both passes, and
`deep_weekly_workflow` barely moved (0% → 10%, still mostly absorbed by reject). These
confusions require the model to actually *reason* about the request — recognizing "yesterday"
as a disqualifying past date, or recognizing a missing detail as grounds for a clarifying
question rather than an outright decline — and a smarter prompt can't teach that reasoning if
the training examples themselves don't clearly demonstrate it.

**Conclusion:** this is a data problem, not an instructions problem, for the remaining three
classes. The next lever is more and better-contrasted training examples for
`fast_path_reject`, `ambiguous_clarify`, and `deep_weekly_workflow` specifically — which is
what the `CONFUSABLE_WITH`-guided regeneration in `tools/generate_dataset.py` targets next.

### Where this could go next

1. **Regenerate the dataset** with `tools/generate_dataset.py`'s contrastive `CONFUSABLE_WITH`
   guidance (added after pass 1) and retrain — same prompt as pass 2, new data — to test
   whether the reject-cluster confusion is fixable with better-contrasted examples where
   better instructions alone were not.
2. **Production integration**: wire the merged model in as a cheap pre-filter ahead of the
   Family Coordinator's own routing reasoning in the family-calendar project — call it first,
   and only fall back to the full frontier-model reasoning path when its confidence is low or
   the request lands in `ambiguous_clarify`. Not required for this submission, but it's the
   actual production version of the "small model as a routing gate" idea the reference project
   argues for.